# 02 — ModernBERT con le etichette NewsGuard 2026 (tentativo 11)

La ricetta della Versione 2, invariata (come nel tentativo 10), con l'etichetta scelta da `ETICHETTA`: livello
NewsGuard **2026** (obiettivo) o **2022** (controllo); stesso dataset e stessi fold del classificatore a feature.

- ModernBERT-base, 1024 token, solo il testo dell'articolo; **2 epoche fisse**, lr 2e-5, batch effettivo 32 (batch 4
  con accumulo, sulla GPU condivisa), fp16.
- A ogni epoca al più 5 testate per gruppo (diverse a ogni epoca) e 20 articoli per testata; pesi di classe sulle
  testate pesate. Un 1/7 dei gruppi di training tenuto da parte serve solo al log.
- Per testata: la **media delle probabilità** dei suoi articoli. Un seme per esecuzione (`SEED`): 42, 1 e 2, poi la
  media dei tre in `03_confronto`.
- Se la memoria GPU finisce si riparte con batch dimezzato; prima di ogni fold si aspetta che sulla GPU condivisa ci
  siano almeno `GB_GPU` liberi.

Varianti del weekend: con `TITOLO` il modello legge la coppia *titolo [SEP] articolo* (titolo pulito dalla cornice
del sito, intero; si taglia solo l'articolo; come nella Versione 4); con `SOLO_POLITICI` solo le testate con il solo
argomento politico.

Si salva tutto in `Results/3-classificazione/transformer/<NOME>/` (`articoli.csv` con logit e probabilità,
`metriche.json`, `FATTO`; per fold le epoche e la validazione).


In [1]:
import gc
import json
import math
import random
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

from comune import CLASSI, RESULTS, carica, metriche, per_testata_media, registratore

/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
NOME = "modernbert"
ETICHETTA = "2026"         # 2026 (obiettivo) / 2022 (controllo)
TITOLO = False            # True: la coppia titolo [SEP] articolo
SOLO_POLITICI = False     # True: solo le testate con il solo argomento politico
MODELLO = "answerdotai/ModernBERT-base"
MAX_TOKEN = 1024
BATCH = 4                  # esempi per passo sulla GPU
BATCH_EFFETTIVO = 32       # con accumulo del gradiente
BATCH_EVAL = 16
ARTICOLI_PER_TESTATA = 20
MAX_TESTATE_PER_GRUPPO = 5 # per gruppo, a ogni epoca (tetto a rotazione)
SOLO_VALUTAZIONE = False   # True: training solo sulle testate valutate
EPOCHE_MAX = 2
PATIENCE = 0               # 0 = epoche fisse (EPOCHE_MAX), senza early stopping
LR = 2e-5
WARMUP = 0.1
WEIGHT_DECAY = 0.01
SEED = 42
SCHEMA = "fold_gruppo"
FOLDS = 0                  # 0 = tutti; k = solo il fold k
FP16 = True
GB_GPU = 6.0               # memoria GPU libera (contando quella che il processo ha già per sé) per partire un fold
PROVA = False              # True: poche testate, 1 epoca, 1 fold, risultati in Results/3-classificazione/prova/


In [3]:
# Parameters
ETICHETTA = 2022
TITOLO = False
SOLO_POLITICI = True
SEED = 42


In [4]:
def fissa_seed(s):
    s = int(s)                 # numpy int64 non va bene per random.seed
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


NOME = NOME + ("_titolo" if TITOLO else "") + ("_politici" if SOLO_POLITICI else "") + f"_{ETICHETTA}_s{SEED}"
base = RESULTS / ("prova" if PROVA else "")
OUT = base / "transformer" / NOME
OUT.mkdir(parents=True, exist_ok=True)
registra = registratore(base / f"avanzamento_{NOME}.log")
if PROVA:
    EPOCHE_MAX = 1
config = {k: globals()[k] for k in ["NOME", "MODELLO", "MAX_TOKEN", "BATCH", "BATCH_EFFETTIVO", "BATCH_EVAL", "ARTICOLI_PER_TESTATA",
                                    "MAX_TESTATE_PER_GRUPPO", "SOLO_VALUTAZIONE", "EPOCHE_MAX", "PATIENCE", "LR", "WARMUP", "WEIGHT_DECAY", "SEED",
                                    "SCHEMA", "FP16", "ETICHETTA", "TITOLO", "SOLO_POLITICI", "PROVA"]}
config.update({"torch": torch.__version__, "transformers": __import__("transformers").__version__,
               "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"})
json.dump(config, open(OUT / "config.json", "w"), indent=1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
registra(f"=== {NOME}: {MODELLO}, {MAX_TOKEN} token, seed {SEED}, su {config['gpu']} ===")

art = carica(ETICHETTA, PROVA, SOLO_VALUTAZIONE, titolo=TITOLO, solo_politici=SOLO_POLITICI)
tok = AutoTokenizer.from_pretrained(MODELLO)
t0 = time.time()
if TITOLO:      # la coppia titolo / articolo: si taglia solo l'articolo
    ids = tok(art.titolo.tolist(), art.text.tolist(), truncation="only_second", max_length=MAX_TOKEN)["input_ids"]
else:
    ids = tok(art.text.tolist(), truncation=True, max_length=MAX_TOKEN)["input_ids"]
N_USCITE = 5
lunghezze = np.array([len(x) for x in ids])
registra(f"{len(art)} articoli tokenizzati in {time.time() - t0:.0f} s; token per articolo: mediana {int(np.median(lunghezze))}, "
         f"al massimo {MAX_TOKEN} nel {100 * (lunghezze >= MAX_TOKEN).mean():.0f}% dei casi")
PAD = tok.pad_token_id

2026-10-23 22:23:43  === modernbert_politici_2022_s42: answerdotai/ModernBERT-base, 1024 token, seed 42, su Tesla V100S-PCIE-32GB ===


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


2026-10-23 22:24:50  25403 articoli tokenizzati in 63 s; token per articolo: mediana 909, al massimo 1024 nel 40% dei casi


## Gli strumenti

In [5]:
def lotto(indici):
    """Padding dinamico: ogni lotto è lungo quanto il suo articolo più lungo."""
    L = max(lunghezze[i] for i in indici)
    x = np.full((len(indici), L), PAD, dtype=np.int64)
    m = np.zeros((len(indici), L), dtype=np.int64)
    for r, i in enumerate(indici):
        x[r, :lunghezze[i]] = ids[i]; m[r, :lunghezze[i]] = 1
    return torch.from_numpy(x).to(device), torch.from_numpy(m).to(device)


@torch.no_grad()
def predici(model, indici):
    """Logit (float32) per gli articoli indicati, nello stesso ordine; lotti per lunghezza per ridurre il padding."""
    model.eval()
    indici = np.asarray(indici)
    ordine = np.argsort(lunghezze[indici])
    out = np.zeros((len(indici), 5), dtype=np.float32)
    for s in range(0, len(indici), BATCH_EVAL):
        pezzo = ordine[s:s + BATCH_EVAL]
        x, m = lotto(indici[pezzo])
        with torch.autocast("cuda", dtype=torch.float16, enabled=FP16):
            out[pezzo] = model(input_ids=x, attention_mask=m).logits[:, :5].float().cpu().numpy()
    return out


def tabella(indici, logit):
    sotto = art.loc[indici]
    p = torch.softmax(torch.from_numpy(logit), 1).numpy()
    t = pd.DataFrame({"article_id": sotto.article_id.values, "domain": sotto.domain.values, "fold": sotto[SCHEMA].values,
                      "y_true": sotto.label.values})
    for k in range(5):
        t[f"logit_l{k + 1}"] = logit[:, k]
    for k, c in enumerate(CLASSI):
        t[c] = p[:, k]
    t["y_pred"] = p.argmax(1)
    return t


def f1_testate(t):
    tm = per_testata_media(t)
    return f1_score(tm.y_true, tm.y_pred, average="macro", labels=range(5), zero_division=0)

## Un fold

In [6]:
def un_fold(fold):
    fissa_seed(SEED + fold)
    t_fold = time.time()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    te = art.index[(art[SCHEMA] == fold) & art.valutazione]
    tr_tutti = art[art[SCHEMA] != fold]
    testate_tr = tr_tutti.drop_duplicates("domain").reset_index(drop=True)
    _, iv = next(StratifiedGroupKFold(7, shuffle=True, random_state=SEED).split(testate_tr, testate_tr.label, testate_tr.gruppo))
    in_val = set(testate_tr.domain.iloc[iv])                      # gruppi interi in validazione (solo per il log)
    tr = tr_tutti[~tr_tutti.domain.isin(in_val)]
    va = tr_tutti.index[tr_tutti.domain.isin(in_val) & tr_tutti.valutazione]
    articoli_di = {d: g.index.to_numpy() for d, g in tr.groupby("domain")}
    testate_di = {g: sorted(x.domain.unique()) for g, x in tr.groupby("gruppo")}
    t_tr = tr.drop_duplicates("domain")
    conta = t_tr.groupby("label").peso.sum().reindex(range(5), fill_value=0).to_numpy()   # testate pesate: un gruppo vale al più 5
    pesi = torch.tensor(conta.sum() / (5 * np.maximum(conta, 1e-9)), dtype=torch.float32, device=device)

    def campione_epoca(epoca):
        """Al più MAX_TESTATE_PER_GRUPPO testate per gruppo (diverse a ogni epoca), ARTICOLI_PER_TESTATA articoli ciascuna."""
        rng = np.random.default_rng(SEED * 1000 + fold * 100 + epoca)
        pezzi = []
        for g in sorted(testate_di):
            ds = testate_di[g]
            for d in (rng.choice(ds, MAX_TESTATE_PER_GRUPPO, replace=False) if len(ds) > MAX_TESTATE_PER_GRUPPO else ds):
                pezzi.append(rng.choice(articoli_di[d], min(len(articoli_di[d]), ARTICOLI_PER_TESTATA), replace=False))
        c = np.concatenate(pezzi)
        rng.shuffle(c)
        return c

    campioni = {e: campione_epoca(e) for e in range(1, EPOCHE_MAX + 1)}
    registra(f"fold {fold}: {len(articoli_di)} testate di training ({t_tr.valutazione.sum()} valutate) in {len(testate_di)} gruppi, "
             f"{len(in_val)} di validazione, {art.loc[te].domain.nunique()} di test; {len(campioni[1])} articoli per epoca; "
             f"pesi di classe {np.round(pesi.cpu().numpy(), 2).tolist()}; batch {BATCH}")

    kw = {"reference_compile": False} if "ModernBERT" in MODELLO else {}
    model = AutoModelForSequenceClassification.from_pretrained(MODELLO, num_labels=N_USCITE, **kw).to(device)
    nodecay = ["bias", "LayerNorm.weight", "norm.weight"]
    gruppi = [{"params": [p for n, p in model.named_parameters() if not any(x in n for x in nodecay)], "weight_decay": WEIGHT_DECAY},
              {"params": [p for n, p in model.named_parameters() if any(x in n for x in nodecay)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(gruppi, lr=LR)
    accumulo = max(1, BATCH_EFFETTIVO // BATCH)
    passi_totali = sum(math.ceil(len(c) / (BATCH * accumulo)) for c in campioni.values())
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * passi_totali), passi_totali)
    scaler = torch.amp.GradScaler("cuda", enabled=FP16)
    loss_fn = torch.nn.CrossEntropyLoss(weight=pesi)
    migliore, stato_migliore, logit_val_migliori, attese, righe = (-1.0, np.inf), None, None, 0, []

    for epoca in range(1, EPOCHE_MAX + 1):
        t0 = time.time()
        campione = campioni[epoca]
        model.train(); tot, n = 0.0, 0
        opt.zero_grad()
        for s in range(0, len(campione), BATCH):
            pezzo = campione[s:s + BATCH]
            x, m = lotto(pezzo)
            y = torch.tensor(art.label.values[pezzo], device=device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=FP16):
                logit = model(input_ids=x, attention_mask=m).logits.float()
            loss = loss_fn(logit[:, :5], y)
            scaler.scale(loss / accumulo).backward()
            tot += loss.item() * len(pezzo); n += len(pezzo)
            if (s // BATCH + 1) % accumulo == 0 or s + BATCH >= len(campione):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
        logit_val = predici(model, va)
        t_val = tabella(va, logit_val)
        loss_val = float(torch.nn.functional.cross_entropy(torch.from_numpy(logit_val), torch.tensor(art.label.values[va])).item())
        f1_val = f1_testate(t_val)
        acc_val = float((t_val.y_true == t_val.y_pred).mean())
        righe.append({"fold": fold, "epoca": epoca, "loss_train": tot / n, "loss_val": loss_val, "acc_val_articoli": acc_val,
                      "f1_val_testate": f1_val, "esempi": n, "secondi": time.time() - t0})
        registra(f"fold {fold} epoca {epoca}/{EPOCHE_MAX}: loss {tot / n:.4f}, val loss {loss_val:.4f}, val F1 testate {f1_val:.3f}, "
                 f"val acc articoli {acc_val:.3f} ({time.time() - t0:.0f} s)")
        if PATIENCE == 0:                    # epoche fisse: niente early stopping, si tiene l'ultima epoca
            logit_val_migliori = logit_val
            continue
        if (f1_val, -loss_val) > (migliore[0], -migliore[1]):
            migliore, attese = (f1_val, loss_val), 0
            stato_migliore = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            logit_val_migliori = logit_val
        else:
            attese += 1
            if attese >= PATIENCE:
                registra(f"fold {fold}: stop all'epoca {epoca}, la migliore è la {righe[-1 - attese]['epoca']}")
                break
    if stato_migliore is not None:
        model.load_state_dict(stato_migliore)
    t_test = tabella(te, predici(model, te))
    pd.DataFrame(righe).to_csv(OUT / f"fold{fold}_epoche.csv", index=False)
    tabella(va, logit_val_migliori).to_csv(OUT / f"fold{fold}_val.csv", index=False)
    t_test.to_csv(OUT / f"fold{fold}_test.csv", index=False)       # per ultimo: la sua presenza vuol dire "fold finito"
    registra(f"fold {fold}: finito in {(time.time() - t_fold) / 60:.0f} min, test acc articoli {(t_test.y_true == t_test.y_pred).mean():.3f}, "
             f"F1 testate {f1_testate(t_test):.3f}, memoria GPU massima {torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0:.1f} GB")
    del model, opt, sched, scaler, stato_migliore
    gc.collect(); torch.cuda.empty_cache()

## Tutti i fold

In [7]:
folds = [int(FOLDS)] if FOLDS else [int(f) for f in sorted(art[SCHEMA].unique())]
if PROVA:
    folds = folds[:1]


def aspetta_gpu():
    while torch.cuda.is_available():
        libera = (torch.cuda.mem_get_info()[0] + torch.cuda.memory_reserved()) / 2**30     # anche quella che ha già questo processo
        if libera >= GB_GPU:
            return
        registra(f"GPU occupata da altri: {libera:.1f} GB liberi, ne servono {GB_GPU}; riprovo tra 5 minuti")
        time.sleep(300)


for fold in folds:
    if (OUT / f"fold{fold}_test.csv").exists():
        registra(f"fold {fold}: già fatto, salto")
        continue
    while True:
        aspetta_gpu()
        memoria_finita = False
        try:
            un_fold(fold)
        except RuntimeError as e:                  # OutOfMemoryError, ma anche CUDA/cuBLAS che non trovano memoria;
            # su artes4 NVML è rotto (driver/library mismatch): quando la memoria finisce PyTorch chiede i dettagli a NVML
            # e invece di OutOfMemoryError esce "NVML_SUCCESS == ... INTERNAL ASSERT FAILED ... CUDACachingAllocator"
            if not any(x in str(e) for x in ["out of memory", "CUBLAS_STATUS_ALLOC_FAILED", "CUDA error", "CUDACachingAllocator", "NVML"]):
                raise
            registra(f"fold {fold}: errore di memoria GPU: {str(e)[:200]}")
            memoria_finita = True                  # si libera la memoria fuori dall'except, quando il traceback non c'è più
        if not memoria_finita:
            break
        gc.collect(); torch.cuda.empty_cache()
        if BATCH == 1:
            raise RuntimeError("memoria GPU insufficiente anche con batch 1")
        BATCH, BATCH_EVAL = max(1, BATCH // 2), max(1, BATCH_EVAL // 2)
        registra(f"fold {fold}: memoria GPU insufficiente (server condiviso), riparto con batch {BATCH} "
                 f"(stesso batch effettivo {BATCH_EFFETTIVO}) e batch di valutazione {BATCH_EVAL}")
pred = pd.concat([pd.read_csv(OUT / f"fold{f}_test.csv") for f in sorted(art[SCHEMA].unique()) if (OUT / f"fold{f}_test.csv").exists()])
if len(pred) == int(art.valutazione.sum()):
    pred.to_csv(OUT / "articoli.csv", index=False)
    m = metriche(pred)
    json.dump(m, open(OUT / "metriche.json", "w"), indent=1)
    (OUT / "FATTO").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))
    registra(f"=== {NOME} FINITO: testata (media) F1 {m['testata_media']['f1_macro']:.3f}, acc {m['testata_media']['accuratezza']:.3f}, "
             f"bin {m['testata_media']['bin_accuratezza']:.3f} ===")
    display(pd.DataFrame({k: {x: v[x] for x in ["accuratezza", "f1_macro", "bin_accuratezza", "tolerant_0.5", "tolerant_1"]} for k, v in m.items()}).T.round(3))

2026-10-23 22:24:50  fold 1: 115 testate di training (95 valutate) in 89 gruppi, 21 di validazione, 30 di test; 2097 articoli per epoca; pesi di classe [1.350000023841858, 0.8999999761581421, 1.5099999904632568, 0.5799999833106995, 1.3200000524520874]; batch 4


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


2026-10-23 22:28:16  fold 1 epoca 1/2: loss 1.5476, val loss 1.6930, val F1 testate 0.124, val acc articoli 0.181 (204 s)


2026-10-23 22:31:42  fold 1 epoca 2/2: loss 1.3696, val loss 1.6276, val F1 testate 0.160, val acc articoli 0.217 (206 s)


...

2026-10-23 22:59:47  fold 5 epoca 1/2: loss 1.5035, val loss 1.6411, val F1 testate 0.203, val acc articoli 0.276 (205 s)


2026-10-23 23:03:30  fold 5 epoca 2/2: loss 1.2797, val loss 1.7384, val F1 testate 0.190, val acc articoli 0.252 (222 s)


2026-10-23 23:04:22  fold 5: finito in 8 min, test acc articoli 0.304, F1 testate 0.300, memoria GPU massima 5.1 GB


2026-10-23 23:04:23  === modernbert_politici_2022_s42 FINITO: testata (media) F1 0.288, acc 0.336, bin 0.629 ===


,accuratezza,f1_macro,bin_accuratezza,tolerant_0.5,tolerant_1
articolo,0.278,0.254,0.607,0.459,0.640
testata_voto,0.329,0.281,0.636,0.500,0.671
testata_media,0.336,0.288,0.629,0.503,0.671
